In [1]:
import os
os.environ["TQDM_DISABLE"] = "True"

from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from transformers.utils.logging import disable_progress_bar
import transformers
import torch

disable_progress_bar()
transformers.logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

def complete_verse(prompt_text, temperature = 0.7, top_k = 50, top_p = 0.9):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=60,
        no_repeat_ngram_size=2,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

/home/mothyque/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

dataset = load_dataset("biglam/gutenberg-poetry-corpus", split="train")
large_dataset = dataset.select(range(100000))

def tokenize_function(examples):
    return tokenizer(examples["line"], truncation=True, max_length=128)

tokenized_datasets = large_dataset.map(tokenize_function, batched=True, remove_columns=large_dataset.column_names, num_proc=4)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt-poetry-melk",
    num_train_epochs=5,              
    per_device_train_batch_size=12,   

    gradient_accumulation_steps=8,   
    optim="adamw_torch_fused",       
    learning_rate=2e-5,               
    lr_scheduler_type="cosine",     
    warmup_ratio=0.05,                

    fp16=True,                       
    dataloader_num_workers=4,        
    dataloader_pin_memory=True,
    
    save_strategy="epoch",            
    save_total_limit=3,               
    logging_steps=50,
    weight_decay=0.1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

# trainer.train()
# trainer.save_model("./gpt-poetry-melk")

In [3]:
model.from_pretrained("./gpt-poetry-melk").to(device)

print("\n === Testing Fine-Tuned Model ===")
text_fine = complete_verse("The dark waves crash against the silent shore,")
print(text_fine)

print ("\n" + "-" * 30)
model.from_pretrained("gpt2").to(device)
print ("=== Testing Base Model ===")
text_base = complete_verse("The dark waves crash against the silent shore,")
print(text_base)


 === Testing Fine-Tuned Model ===
The dark waves crash against the silent shore, and the sand that covers it fades, leaving a thick, hard, white glow.

The sand is a dense, sticky, black substance. It is what is called "magnitude-locked." It must be completely free of water. The

------------------------------
=== Testing Base Model ===
The dark waves crash against the silent shore, and the moon shines out. It is the night. The moon's light shines on the dark side of the mountains, but it is in vain. A great white cloud comes down upon the people of this world, which they call the Moon. And the


In [4]:
import evaluate
bleu = evaluate.load("bleu")

reference = ["The dark waves crash against the silent shore, And all the stars are watching from on high."]

prediction = [text_fine]
results = bleu.compute(predictions=prediction, references=reference)

print(f"BLEU Score for Fine-Tuned Model: {results['bleu']}")

prediction = [text_base]
results = bleu.compute(predictions=prediction, references=reference)
print(f"BLEU Score for Base Model: {results['bleu']}")

BLEU Score for Fine-Tuned Model: 0.14577198745689388
BLEU Score for Base Model: 0.1414115672222255


In [5]:
print("\n === Testing Base Model on romanian text ===")
text_base = complete_verse("Somnoroase pasarele, pe la cuiburi se aduna,")
print(text_base)

model.from_pretrained("./gpt-poetry-melk").to(device)

print("\n === Testing Fine-Tuned Model on romanian text ===")
text_fine = complete_verse("Somnoroase pasarele, pe la cuiburi se aduna,")
print(text_fine)

# C2: Modelul se descurca cel mai bine, fiind antrenat pe fraze in limba engleza
# C3: Modelul trece prin linguistic drift. Romana este o limba diferita, iar modelul are dificultati in a genera texte coerente in aceasta limba si incearca sa se adapteze, generand texte in limbi asemanatoare (franceza, italiana, spaniola)
# C4: Acelasi lucru ca la c3, apare un conflict de limba, modelul cunoscand doar limba engleza are dificultati in a se adapta la cerinta in romana, nu doar de intelegere ci si de generare.


 === Testing Base Model on romanian text ===
Somnoroase pasarele, pe la cuiburi se aduna, se se n'est pasable, nous ne vous vos, qu'il avez ou plus.

In the evening, I am sitting by the river on the edge of the

 === Testing Fine-Tuned Model on romanian text ===
Somnoroase pasarele, pe la cuiburi se aduna,

Si quia pello, la vida, quod aliquando
, tu si sunt a segundo, sed se gente.
. . .



Lav


In [6]:
pastel_instruction = """ Style: A beautiful nature pastel poem, focusing on the bright colors of the landscape.
Verse:
The golden sun descends upon the emerald hill,
"""

print("\n === Testing Fine-Tuned Model on Pastel Poem ===")
pastel_poez = complete_verse(pastel_instruction)
print(pastel_poez)

model.from_pretrained("./gpt-poetry-melk").to(device)
print("\n === Testing Base Model on Pastel Poem ===")
text_base = complete_verse(pastel_instruction)
print(text_base)


 === Testing Fine-Tuned Model on Pastel Poem ===
 Style: A beautiful nature pastel poem, focusing on the bright colors of the landscape.
Verse:
The golden sun descends upon the emerald hill,
And the moon rises above the sea.


Jump to Previous

Next

, or to the right. The sun

 === Testing Base Model on Pastel Poem ===
 Style: A beautiful nature pastel poem, focusing on the bright colors of the landscape.
Verse:
The golden sun descends upon the emerald hill,
And the sun rises on it, and the stars shine upon it.


Jump to Previous

Next
. .
